In [ ]:
!pip install torch

In [ ]:
!pip install shap

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
import math
import time
import json
import os
import matplotlib.pyplot as plt
from collections import deque
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.impute import SimpleImputer
from torch.utils.data import TensorDataset, DataLoader
import copy
import joblib
import gc
import shap
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

In [ ]:
# Data Preprocessing Class
class preprocess_data():
    def __init__(self):
        self.file_path = None
        self.le = {}
        self.ss = {}
        self.feature_selecter = {}
        self.df = []
        self.num_cols = []
        self.cat_cols = []
        self.features = []
        self.target = ""
        self.clean_df = []
        self.X = []
        self.y = []
        self.X_train = []
        self.X_test = []
        self.y_train = []
        self.y_test = []

    def load_csv(self, file_path):
        self.file_path = file_path
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"Dataset file '{self.file_path}' not found. Please ensure the file exists.")

        try:
            print(f"Loading dataset from {self.file_path}...")
            self.df = pd.read_csv(self.file_path)
            print(f"Dataset loaded successfully. Shape: {self.df.shape}")
        except Exception as e:
            raise Exception(f"Error loading dataset: {str(e)}")

    def handling_missing_values(self, file_path):
        self.load_csv(file_path)

        # Set features and target
        self.features = self.df.columns[:-1].tolist()
        self.target = self.df.columns[-1]

        # Identify types
        self.num_cols = self.df[self.features].select_dtypes(include=np.number).columns.tolist()
        self.cat_cols = self.df[self.features].select_dtypes(include=['object']).columns.tolist()

        # Handle infinite values
        for col in self.df.columns:
            if pd.api.types.is_numeric_dtype(self.df[col]):
                mean_val = self.df[col][np.isfinite(self.df[col])].mean()
                self.df[col] = self.df[col].replace([np.inf, -np.inf], mean_val)

        # Impute numerical
        for col in self.num_cols:
            if col == self.target:
                continue
            if self.df[col].isnull().sum() > 0:
                imputer = SimpleImputer(strategy='mean')
                self.df[col] = imputer.fit_transform(self.df[[col]])

        # Impute categorical
        for col in self.cat_cols:
            if col == self.target:
                continue
            if self.df[col].isnull().sum() > 0:
                imputer = SimpleImputer(strategy='most_frequent')
                self.df[col] = imputer.fit_transform(self.df[[col]])

        return self.df, self.features, self.target


    def scaling_encoding(self):
        # Recalculate categorical and numeric cols from training data
        self.num_cols = self.df[self.features].select_dtypes(include=np.number).columns.tolist()
        self.cat_cols = self.df[self.features].select_dtypes(include=['object']).columns.tolist()

        high_cardinality_cols = []
        low_cardinality_cols = []

        for col in self.cat_cols:
            unique_vals = self.df[col].nunique()
            if unique_vals > 10:
                high_cardinality_cols.append(col)
            else:
                low_cardinality_cols.append(col)

        # --- Label encode high-cardinality ---
        for col in high_cardinality_cols:
            le = LabelEncoder()
            self.df[col] = le.fit_transform(self.df[col].astype(str))
            self.le[col] = le

        # --- One-hot encode low-cardinality ---
        self.df = pd.get_dummies(self.df, columns=low_cardinality_cols, drop_first=True)

        # --- Update numerical columns after encoding ---
        self.num_cols = self.df.select_dtypes(include=np.number).columns.tolist()

        # --- Standard scaling ---
        for col in self.num_cols:
            scaler = StandardScaler()
            self.df[col] = scaler.fit_transform(self.df[[col]])
            self.ss[col] = scaler

        # Final features
        self.features = self.df.columns.tolist()

        return self.df, self.features, self.target

    def handling_outliers(self):
        for col in self.num_cols:
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            outliers = (self.df[col] < lower_bound) | (self.df[col] > upper_bound)
            num_outliers = outliers.sum()

            if num_outliers > 0:
                median_val = self.df[col].median()
                self.df.loc[outliers, col] = median_val

        return self.df

    def feature_selection(self, k=10):
        # Ensure target is numeric
        if self.df[self.target].dtype == 'object':
            self.df[self.target] = LabelEncoder().fit_transform(self.df[self.target])

        # Remove constant features
        vt = VarianceThreshold(threshold=0.0)
        X_vt = vt.fit_transform(self.df[self.features])
        self.features = [col for col, keep in zip(self.features, vt.get_support()) if keep]

        # Ensure features still exist in df
        self.features = [f for f in self.features if f in self.df.columns]

        # Select K best features
        selector = SelectKBest(score_func=f_classif, k=min(k, len(self.features)))
        X_new = selector.fit_transform(self.df[self.features], self.df[self.target])
        mask = selector.get_support()
        selected_features = [feat for feat, keep in zip(self.features, mask) if keep]

        # Save selector and update feature list
        for col in selected_features:
            self.feature_selecter[col] = selector
        self.features = selected_features

        # Update df to include only selected features + target
        self.df = self.df[selected_features + [self.target]]

        return self.features

    def train_split(self):
        # Prepare final cleaned DataFrame
        self.clean_df = self.df[self.features + [self.target]]

        # Final X and y (entire dataset, not split)
        self.X = self.df[self.features]
        self.y = self.df[self.target]

        print("Cleaned Data Shape: ", self.clean_df.shape)
        print("X Shape: ", self.X.shape)
        print("y Shape: ", self.y.shape)

        return self.clean_df, self.X, self.y

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class MiniTransformerAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=32, n_heads=2, ff_dim=64, n_layers=2, seq_len=10):
        super(MiniTransformerAutoencoder, self).__init__()
        self.seq_len = seq_len
        self.input_dim = input_dim
        self.embedding = nn.Linear(input_dim, ff_dim)
        self.pos_encoder = PositionalEncoding(ff_dim, max_len=seq_len)
        self.classifier = nn.Linear(latent_dim, 2)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=ff_dim,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=0.1,
            activation='relu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.encoder_fc = nn.Linear(ff_dim, latent_dim)
        self.decoder_fc = nn.Linear(latent_dim, ff_dim)
        self.output_layer = nn.Linear(ff_dim, input_dim)
    def get_weights(self):
        return {k: v.cpu().detach().clone() for k, v in self.state_dict().items()}

    def set_weights(self, weights):
        self.load_state_dict(weights)

    def forward(self, x):
        # Encode
        x = self.embedding(x)                     # (batch, seq_len, ff_dim)
        x = self.pos_encoder(x)                   # Add positional encoding
        x = self.transformer_encoder(x)           # Transformer encoder
        latent_seq = self.encoder_fc(x)           # (batch, seq_len, latent_dim)

        # Use the last timestep as summary representation
        latent_vector = latent_seq[:, -1, :]      # (batch, latent_dim)

        # Classification output
        logits = self.classifier(latent_vector)   # (batch, 2)

        # Decode (broadcast latent back over time)
        repeated_latent = latent_vector.unsqueeze(1).repeat(1, self.seq_len, 1)  # (batch, seq_len, latent_dim)
        x = self.decoder_fc(repeated_latent)      # (batch, seq_len, ff_dim)
        reconstructed = self.output_layer(x)      # (batch, seq_len, input_dim)

        return reconstructed, logits


In [ ]:
# Base class for all nodes in the IoT architecture
class Node:
    def __init__(self, node_id, name):
        self.node_id = node_id
        self.node_name = name
        self.parent = None
        self.children = []
        self.data_buffer = []
        self.last_update_time = time.time()

    def add_child(self, child):
        self.children.append(child)
        child.parent = self

    def send_data(self, data, destination):
        print(f"\n[{self.node_name}] Sending data to {destination.node_name}")
        destination.receive_data(data, self)

    def receive_data(self, data, source):
        print(f"\n[{self.node_name}] Received data from {source.node_name}\n")
        self.data_buffer.append(data)
        self.last_update_time = time.time()

    def evaluate_model(self, model, test_loader, device):
        model.to(device)
        model.eval()
        correct, total = 0, 0
        total_loss = 0
        criterion = nn.CrossEntropyLoss()

        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                _, logits = model(X)

                # Classification accuracy
                _, predicted = torch.max(logits, dim=1)
                total += y.size(0)
                correct += (predicted == y).sum().item()

                # Optional: Track classification loss
                total_loss += criterion(logits, y).item()

        accuracy = correct / total
        avg_loss = total_loss / len(test_loader)

        print(f"Accuracy: {accuracy:.4f}")
        print(f"Classification Loss: {avg_loss:.4f}")


    def create_test_sequences(self, X, seq_len=10):
        sequences = []
        for i in range(len(X) - seq_len):
            sequences.append(X[i:i+seq_len])
        return np.array(sequences)

    def safe_convert_sequences(self, X_seq):
        try:
            X_seq = X_seq.astype(np.float32)
        except ValueError:
            # Fall back: clean manually
            cleaned_seq = []
            for seq in X_seq:
                try:
                    cleaned_seq.append(seq.astype(np.float32))
                except (ValueError, TypeError):
                    continue  # Drop this sequence
            X_seq = np.array(cleaned_seq, dtype=np.float32)
        return X_seq

    def process_data(self):
        raise NotImplementedError("Subclasses must implement process_data()")



class Edge(Node):
    def __init__(self, node_id, name, local_df):
      super().__init__(node_id, name)
      self.df = local_df
      self.X = self.df.drop(columns=["Label"]).values
      self.y = self.df["Label"].values

      self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
      self.sequence_length = 10
      self.epochs = 3
      self.local_model = None
      self.global_model_weights = None
      self.local_model_weights = None
      self.input_dim = None

    def recieve_global_model(self, global_model_weights):
      print(f"[{self.node_name}] Received global model from Gateway.")
      self.global_model_weights = global_model_weights

    def process_data(self):

      self.input_dim = self.X.shape[1]
      print(self.input_dim)

      if self.global_model_weights:
          self.local_model = MiniTransformerAutoencoder(
              input_dim=self.input_dim,
              latent_dim=32,
              ff_dim=64,
              n_heads=2,
              n_layers=2,
              seq_len=self.sequence_length
          ).to(self.device)
          self.local_model.set_weights(self.global_model_weights) # Loading global model
          print(f"\n[Sensor {self.node_id}] Loaded global model for fine-tuning.")
      else:
          self.local_model = MiniTransformerAutoencoder(
              input_dim=self.input_dim,
              latent_dim=32,
              ff_dim=64,
              n_heads=2,
              n_layers=2,
              seq_len=self.sequence_length
          ).to(self.device)
          print(f"[Sensor {self.node_id}] Training from scratch.") # Training local model

      print(f"Sensor: {self.node_id} started training!\n")

      # Filter only benign samples
      benign_indices = self.y == 0  # assuming 0 = benign, 1 = attack
      benign_X = self.X[benign_indices]

      # Generate sequences only from benign data
      X_seq = self.create_sequences(benign_X, self.sequence_length)
      X_seq = self.safe_convert_sequences(X_seq)
      X_tensor = torch.tensor(X_seq, dtype=torch.float32).to(self.device)

      print(f"[DEBUG] Input tensor stats: min={X_tensor.min().item():.4f}, max={X_tensor.max().item():.4f}, mean={X_tensor.mean().item():.4f}")
      print(f"[DEBUG] Input tensor shape: {X_tensor.shape}")

      # Training starts
      self.train_local_model(X_tensor, self.epochs)

    def create_sequences(self, data, seq_len):
      sequences = []
      for i in range(len(data) - seq_len):
          seq = data[i:i+seq_len]
          sequences.append(seq)
      return np.array(sequences)

    def train_local_model(self, X_tensor, epochs=3):
      self.local_model.train()
      optimizer = torch.optim.Adam(self.local_model.parameters(), lr=0.001)
      criterion = nn.MSELoss()

      for epoch in range(epochs):
          output, _ = self.local_model(X_tensor)
          loss = criterion(output, X_tensor)

          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

          if epoch % 1 == 0:
              print(f"[Sensor {self.node_id}] Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")
      print(f"[Sensor {self.node_id}] Final Local Loss: {loss.item():.4f}")


      self.local_model_weights = self.local_model.get_weights()

    def send_model_weights(self):
      if self.parent:
          packet = {
              'node_id': self.node_id,
              'weights': self.local_model_weights
          }
          self.send_data(packet, self.parent)


class Gateway(Node):
    def __init__(self, node_id, name):
        super().__init__(node_id, name)
        self.received_weights = None
        self.global_model_weights = None
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.sequence_length = 10
        self.input_dim = 87
        self.local_model = None
        self.local_model_weights = None

    def receive_data(self, data, source):
        super().receive_data(data, source)
        if 'weights' in data:
            self.received_weights = data['weights']

    def process_data(self):
        if self.parent:
            packet = {
                'node_id': self.node_id,
                'weights': self.received_weights
            }
            self.send_data(packet, self.parent)

    def receive_global_model(self, global_weights):
        self.global_model_weights = global_weights
        print(f"[{self.node_name}] Received global model from Fog.")
        for edge in self.children:
            edge.recieve_global_model(global_weights)

    def eval(self, file_path):
        print(f"[{self.node_name}] Testing the local model received from edge devices!")

        if not self.received_weights:
            print(f"[{self.node_name}] No model weights available for testing!")
            return

        self.file_path = file_path
        # Load and preprocess data using the updated preprocess_data class
        self.dp = preprocess_data()

        # Step 1: Handle missing values and load dataset
        self.df, self.features, self.target = self.dp.handling_missing_values(self.file_path)

        # Step 2: Encode and scale features
        self.df, self.features, self.target = self.dp.scaling_encoding()

        # Step 3: Handle outliers
        self.df = self.dp.handling_outliers()

        # Step 4: Feature selection
        self.dp.feature_selection(k=10)  # You can change k as needed

        # Step 5: Split into X and y
        self.clean_df, self.X, self.y = self.dp.train_split()

        # Get actual input dimension
        self.input_dim = self.X.shape[1]
        print(f"[{self.node_name}] Input dimension: {self.input_dim}")

        # sample_size = min(100020, len(X))
        X_sampled = self.X
        y_sampled = self.y
        y_sampled = np.array(y_sampled)

        # Convert and generate sequences
        X_seq = self.create_test_sequences(X_sampled.values, seq_len=10)
        y_seq = y_sampled[10:]

        X_seq = self.safe_convert_sequences(X_seq)
        y_seq = self.safe_convert_sequences(y_seq)

        X_test_tensor = torch.tensor(X_seq, dtype=torch.float32)
        y_test_tensor = torch.tensor(np.array(y_seq), dtype=torch.long)

        test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
        test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, pin_memory=True)

        # Create model with correct input dimension
        self.local_model = MiniTransformerAutoencoder(
            input_dim=self.input_dim,
            latent_dim=32,
            ff_dim=64,
            n_heads=2,
            n_layers=2,
            seq_len=self.sequence_length
        ).to(self.device)

        # Load the received weights
        self.local_model.set_weights(self.received_weights)
        print(f"[{self.node_name}] Model loaded successfully!")

        self.evaluate_model(self.local_model, test_loader, self.device)

class Fog(Node):
    def __init__(self, node_id, name):
        super().__init__(node_id, name)
        self.global_model_weights = None
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.global_model = None
        self.input_dim = 87
        self.sequence_length = 10
        self.received_weights = None

    def receive_data(self, data, source):
        super().receive_data(data, source)
        if isinstance(data, list):
            self.received_weights = data
        elif isinstance(data, dict) and 'weights' in data:
            self.received_weights = data['weights']
        else:
            self.received_weights = []

    def aggregate_weights(self, weights_list):
        if isinstance(weights_list, dict):
            weights_list = [weights_list]

        if not weights_list or not isinstance(weights_list[0], dict):
            raise ValueError(f"[{self.node_name}] Invalid weight list received for aggregation: {type(weights_list)}")

        avg_weights = copy.deepcopy(weights_list[0])
        for k in avg_weights:
            for i in range(1, len(weights_list)):
                avg_weights[k] += weights_list[i][k]
            avg_weights[k] /= len(weights_list)
        return avg_weights

    def process_data(self):
        if not hasattr(self, 'received_weights') or not self.received_weights:
            print(f"[{self.node_name}] No weights to aggregate.")
            return

        self.global_model_weights = self.aggregate_weights(self.received_weights)
        print(f"[{self.node_name}] Global model aggregated. Now sent to Edge nodes.\n")

        # Get input dimension from the first layer of aggregated weights
        # This assumes the embedding layer is named 'embedding.weight'
        for key in self.global_model_weights.keys():
            if 'embedding.weight' in key:
                self.input_dim = self.global_model_weights[key].shape[1]
                break

        if self.input_dim is None:
            print(f"[{self.node_name}] Warning: Could not determine input dimension, using default 78")
            self.input_dim = 78

        self.global_model = MiniTransformerAutoencoder(
            input_dim=self.input_dim,
            latent_dim=32,
            ff_dim=64,
            n_heads=2,
            n_layers=2,
            seq_len=self.sequence_length
        ).to(self.device)

        self.global_model.load_state_dict(self.global_model_weights)
        torch.save(self.global_model.state_dict(), "Global_Model_Fog.pth")

        for gateway in self.children:
            if hasattr(gateway, "receive_global_model"):
                gateway.receive_global_model(self.global_model_weights)

    def recieve_global_model(self, global_model_weights):
        self.global_model_weights = global_model_weights
        for gateway in self.children:
            if hasattr(gateway, "receive_global_model"):
                gateway.receive_global_model(self.global_model_weights)

    def send_global_weights(self):
        if self.parent:
            packet = {
                'node_id': self.node_id,
                'weights': self.global_model_weights
            }
            self.send_data(packet, self.parent)

    def eval(self, file_path):
        print(f"[{self.node_name}] Testing the aggregated global model!")

        if not self.global_model_weights:
            print(f"[{self.node_name}] No global model available for testing!")
            return

        self.file_path = file_path
        # Load and preprocess data using the updated preprocess_data class
        self.dp = preprocess_data()

        # Step 1: Handle missing values and load dataset
        self.df, self.features, self.target = self.dp.handling_missing_values(self.file_path)

        # Step 2: Encode and scale features
        self.df, self.features, self.target = self.dp.scaling_encoding()

        # Step 3: Handle outliers
        self.df = self.dp.handling_outliers()

        # Step 4: Feature selection
        self.dp.feature_selection(k=10)  # You can change k as needed

        # Step 5: Split into X and y
        self.clean_df, self.X, self.y = self.dp.train_split()

        # Get actual input dimension
        self.input_dim = self.X.shape[1]
        print(f"[{self.node_name}] Input dimension: {self.input_dim}")

        # sample_size = min(100020, len(X))
        X_sampled = self.X
        y_sampled = self.y
        y_sampled = np.array(y_sampled)

        # Convert and generate sequences
        X_seq = self.create_test_sequences(X_sampled.values, seq_len=10)
        y_seq = y_sampled[10:]

        X_seq = self.safe_convert_sequences(X_seq)
        y_seq = self.safe_convert_sequences(y_seq)

        X_test_tensor = torch.tensor(X_seq, dtype=torch.float32)
        y_test_tensor = torch.tensor(np.array(y_seq), dtype=torch.long)

        test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
        test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, pin_memory=True)


        # Recreate model with correct input dimension if needed
        if self.global_model is None or self.global_model.input_dim != self.input_dim:
            self.global_model = MiniTransformerAutoencoder(
                input_dim=self.input_dim,
                latent_dim=32,
                ff_dim=64,
                n_heads=2,
                n_layers=2,
                seq_len=self.sequence_length
            ).to(self.device)
            self.global_model.load_state_dict(self.global_model_weights)

        self.evaluate_model(self.global_model, test_loader, self.device)


class Proxy(Node):
    def __init__(self, node_id, name):
        super().__init__(node_id, name)
        self.received_weights = None
        self.global_model_weights = None
        self.global_model = []


    def receive_data(self, data, source):
        super().receive_data(data, source)
        if 'weights' in data:
            self.received_weights = data['weights']

    def process_data(self):
        if self.parent:
          packet = {
              'node_id': self.node_id,
              'weights': self.received_weights
          }
          self.send_data(packet, self.parent)

    def receive_global_model(self, global_weights):
      self.global_model_weights = global_weights
      print(f"[{self.node_name}] Received global model from Cloud.")
      for fog in self.children:
        print(f"[{fog.node_name}] Received global model from Proxy.")
        fog.recieve_global_model(global_weights)

class Cloud(Node):
    def __init__(self, node_id, name):
        super().__init__(node_id, name)
        self.global_model_weights = None
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.input_dim = 87
        self.sequence_length = 10
        self.global_model = None
        self.received_weights = None

    def receive_data(self, data, source):
        super().receive_data(data, source)
        if isinstance(data, list):
            self.received_weights = data
        elif isinstance(data, dict) and 'weights' in data:
            self.received_weights = data['weights']
        else:
            self.received_weights = []

    def aggregate_weights(self, weights_list):
        if isinstance(weights_list, dict):
            weights_list = [weights_list]

        if not weights_list or not isinstance(weights_list[0], dict):
            raise ValueError(f"[{self.node_name}] Invalid weight list received for aggregation: {type(weights_list)}")

        avg_weights = copy.deepcopy(weights_list[0])
        for k in avg_weights:
            for i in range(1, len(weights_list)):
                avg_weights[k] += weights_list[i][k]
            avg_weights[k] /= len(weights_list)
        return avg_weights

    def process_data(self):
        if not hasattr(self, 'received_weights') or not self.received_weights:
            print(f"[{self.node_name}] No weights to aggregate.")
            return

        self.global_model_weights = self.aggregate_weights(self.received_weights)
        print(f"[{self.node_name}] Global model aggregated. Now sent to Fog nodes.\n")

        for proxy in self.children:
            if hasattr(proxy, "receive_global_model"):
                proxy.receive_global_model(self.global_model_weights)

    def eval(self, file_path):
        print(f"[{self.node_name}] Testing the aggregated global model!")

        if not self.global_model_weights:
            print(f"[{self.node_name}] No global model available for testing!")
            return

        self.file_path = file_path
        # Load and preprocess data using the updated preprocess_data class
        self.dp = preprocess_data()

        # Step 1: Handle missing values and load dataset
        self.df, self.features, self.target = self.dp.handling_missing_values(self.file_path)

        # Step 2: Encode and scale features
        self.df, self.features, self.target = self.dp.scaling_encoding()

        # Step 3: Handle outliers
        self.df = self.dp.handling_outliers()

        # Step 4: Feature selection
        self.dp.feature_selection(k=10)  # You can change k as needed

        # Step 5: Split into X and y
        self.clean_df, self.X, self.y = self.dp.train_split()

        # Get actual input dimension
        self.input_dim = self.X.shape[1]
        print(f"[{self.node_name}] Input dimension: {self.input_dim}")

        # sample_size = min(100020, len(X))
        X_sampled = self.X
        y_sampled = self.y
        y_sampled = np.array(y_sampled)

        # Convert and generate sequences
        X_seq = self.create_test_sequences(X_sampled.values, seq_len=10)
        y_seq = y_sampled[10:]

        X_seq = self.safe_convert_sequences(X_seq)
        y_seq = self.safe_convert_sequences(y_seq)

        X_test_tensor = torch.tensor(X_seq, dtype=torch.float32)
        y_test_tensor = torch.tensor(np.array(y_seq), dtype=torch.long)

        self.global_model = MiniTransformerAutoencoder(
            input_dim=self.input_dim,
            latent_dim=32,
            ff_dim=64,
            n_heads=2,
            n_layers=2,
            seq_len=self.sequence_length
        ).to(self.device)

        # Load the global model weights
        self.global_model.load_state_dict(self.global_model_weights)

        # Save the final global model
        torch.save(self.global_model.state_dict(), "Global_Model_Cloud.pth")
        print(f"[{self.node_name}] Global model saved!")

        test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
        test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, pin_memory=True)

        self.evaluate_model(self.global_model, test_loader, self.device)


class Application(Node):
    def __init__(self, node_id, name, model_path, test_file):
        super().__init__(node_id, name)
        self.model_path = model_path
        self.test_file = test_file
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.X = None
        self.y = None

    def XAI(self):
        self.load_model_and_data()
        self.run_xai()

    def load_model_and_data(self):
        # Load and preprocess data using the updated preprocess_data class
        self.dp = preprocess_data()

        # Step 1: Handle missing values and load dataset
        self.df, self.features, self.target = self.dp.handling_missing_values(self.file_path)

        # Step 2: Encode and scale features
        self.df, self.features, self.target = self.dp.scaling_encoding()

        # Step 3: Handle outliers
        self.df = self.dp.handling_outliers()

        # Step 4: Feature selection
        self.dp.feature_selection(k=10)  # You can change k as needed

        # Step 5: Split into X and y
        self.clean_df, self.X, self.y = self.dp.train_split()

        # sample_size = 100020
        X_sampled = self.X
        y_sampled = self.y

        seq_len = 10

        def create_sequences(X, seq_len):
            return np.array([X[i:i+seq_len] for i in range(len(X) - seq_len)])

        X_seq = create_sequences(X_sampled.values, seq_len)
        y_seq = y_sampled[seq_len:]

        X_seq = self.safe_convert_sequences(X_seq)
        y_seq = self.safe_convert_sequences(y_seq)

        self.X = X_seq
        self.y = y_seq

        input_dim = self.X.shape[2]  # use dim from sequence
        self.model = MiniTransformerAutoencoder(
            input_dim=input_dim,
            latent_dim=32,
            ff_dim=64,
            n_heads=2,
            n_layers=2,
            seq_len=seq_len
        )
        self.model.load_state_dict(torch.load(self.model_path, map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    def run_xai(self):
        if self.model is None or self.X is None:
            print(f"[{self.node_name}] Model or data not loaded. Run `load_model_and_data()` first.")
            return

        print(f"[{self.node_name}] Running XAI")

        test_samples = torch.tensor(self.X, dtype=torch.float32).to(self.device)

        def model_forward(x):
            _, logits = self.model(x)
            return logits

        # SHAP expects inputs of shape (N, seq_len, features)
        explainer = shap.DeepExplainer(model_forward, test_samples)
        shap_values = explainer.shap_values(test_samples)

        # Convert to CPU and numpy for plotting
        sample_to_explain = test_samples[0].detach().cpu().numpy()
        shap_val_to_plot = shap_values[1][0]  # Class 1 SHAP values (if binary classification)

        # Aggregate across time steps if needed
        mean_shap = shap_val_to_plot.mean(axis=0)

        # Get feature names if available
        feature_names = [f"F{i}" for i in range(self.X.shape[2])]

        shap.summary_plot(mean_shap, feature_names=feature_names, plot_type="bar")


In [ ]:
class IOTSimulation:

    def __init__(self, file_path):
        self.file_path = file_path
        preprocessed_splits = []

        # Step 1: Load raw full dataset (no scaling, no encoding!)
        raw_df = pd.read_csv(file_path)
        features = raw_df.columns[:-1].tolist()
        target = raw_df.columns[-1]

        # Step 2: Handle missing values minimally (needed before splitting)
        raw_df.replace([np.inf, -np.inf], np.nan, inplace=True)
        raw_df.dropna(inplace=True)  # Or use simple imputation if needed

        # Step 4: Stratified split on raw data
        splits = IOTSimulation.stratified_split(raw_df, target_column=target, n_splits=2)

        # Step 5: Preprocess each split independently
        for i, split_df in enumerate(splits):
            dp = preprocess_data()
            dp.df = split_df.copy()
            dp.features = split_df.columns[:-1].tolist()
            dp.target = split_df.columns[-1]

            dp.handling_outliers()
            dp.scaling_encoding()
            dp.feature_selection(k=10)
            dp.train_split()

            preprocessed_splits.append(dp)

        # Step 6: Assign preprocessed data to edge nodes
        self.edge_nodes = [
            Edge(i+1, f"Sensor {i+1}", dp.clean_df) for i, dp in enumerate(preprocessed_splits)
        ]


        self.gateway_nodes = [
            Gateway(1,"Bridge 1")
        ]

        self.fog_nodes = [
            Fog(1, "Fog Layer 1")
        ]

        self.proxy_nodes = [
            Proxy(1,"Proxy Layer 1")
        ]

        self.cloud_nodes = [
            Cloud(1,"Cloud Layer 1")
        ]

        self.app_node = Application(1, "Application Layer 1", "Global_Model_Cloud.pth", "/content/wustl_hdrl_2024.csv")


        # Connecting the layers
        for edge in self.edge_nodes:
            self.gateway_nodes[0].add_child(edge)

        self.fog_nodes[0].add_child(self.gateway_nodes[0])

        self.proxy_nodes[0].add_child(self.fog_nodes[0])

        self.cloud_nodes[0].add_child(self.proxy_nodes[0])

        self.app_node.add_child(self.cloud_nodes[0])

    @staticmethod
    def stratified_split(df, target_column, n_splits=2):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        splits = []

        X = df.drop(columns=[target_column])
        y = df[target_column]

        for _, idx in skf.split(X, y):
            split_df = df.iloc[idx].reset_index(drop=True)
            splits.append(split_df)

        return splits

    def run(self):
        gc.collect()
        torch.cuda.empty_cache()
        print("\n========== Federated Round START ==========")

        # edge_threads = []

        # for edge in self.edge_nodes:
        #     thread = Thread(target=edge.process_data)
        #     edge_threads.append(thread)
        #     thread.start()

        # # Wait for all edge nodes to finish training + sending weights
        # for thread in edge_threads:
        #     thread.join()

        # Step 1: Edge training + latent vector sending
        for edge in self.edge_nodes:
            edge.process_data()
            edge.send_model_weights()

        # Step 2: Gateways forward data to Fog
        for gateway in self.gateway_nodes:
            gateway.process_data()  # Send collected vectors to Fog

        # Step 3: Fog processes latent vectors and trains Transformer
        for fog_node in self.fog_nodes:
            fog_node.process_data()
            fog_node.send_global_weights()

        for proxy in self.proxy_nodes:
            proxy.process_data()

        for cloud in self.cloud_nodes:
            cloud.process_data()

        # Saving the Global model
        final_model = self.cloud_nodes[0].global_model
        torch.save(final_model, "Global_Model_Final.pth")
        print("Global model saved!")

        print("========== Federated Round END ==========\n")

    def testing_part(self):
        print("\n========== TESTING PHASE START ==========")

        # Use the same file that was used for training
        test_file_path = '/content/wustl_hdrl_2024.csv' # This ensures consistency

        print(f"\n----- Testing at Gateway Level -----")
        for gateway in self.gateway_nodes:
            gateway.eval(test_file_path)

        gc.collect()
        torch.cuda.empty_cache()

        print(f"\n----- Testing at Fog Level -----")
        for fog in self.fog_nodes:
            fog.eval(test_file_path)

        gc.collect()
        torch.cuda.empty_cache()

        print(f"\n----- Testing at Cloud Level -----")
        for cloud in self.cloud_nodes:
            cloud.eval(test_file_path)

        gc.collect()
        torch.cuda.empty_cache()

        print("\n========== TESTING PHASE END ==========")

    def XAI(self):
        self.app_node.load_model_and_data()
        self.app_node.explain_with_shap()



if __name__ == "__main__":
    sim = IOTSimulation("/content/wustl_hdrl_2024.csv")
    print("\n----- Training -----")
    sim.run()
    print(f"\n----- Testing -----")
    sim.testing_part()

    # print(f"\n----- Application -----")
    # sim.XAI()

Cleaned Data Shape:  (31825, 13)
X Shape:  (31825, 11)
y Shape:  (31825, 2)
Cleaned Data Shape:  (31824, 11)
X Shape:  (31824, 10)
y Shape:  (31824,)

----- Training -----

========== Federated Round START ==========
9
[Sensor 1] Training from scratch.
Sensor: 1 started training!



IndexError: boolean index did not match indexed array along axis 1; size of axis is 9 but size of corresponding boolean axis is 4